# Phase 6: Particle-filter wide-net sweep

The public 7.5-class solutions decode to: **a particle filter tracking the offset
`TVT + Z`** (GR enters as a soft typewell likelihood, never an argmax), optionally
blended with a **beam-search** path and a small **hold-last** weight, with a
**per-well-characteristic selector** choosing the variant. Our faithful port beats
the floor on the dev well (5.12 vs 7.29) — the first method in this project to do so.

This notebook casts the wide net before we commit: PF hyperparameter axes, seed
ensembles, beam configs, hold/beam blends — all scored on the honest 73% harness,
ranked by RMSE on TVT, with bucket splits, win-rates, and an oracle-selector
ceiling over the variants (the input the next notebook's selector needs).

Credit: architecture decoded from the public kernels
`rogii-sel15-forced-selector` / `rogii-wellbore-geology-sp45-fleongg-eda`
(SP45 + Fleongg lineage). Credit them if you publish work derived from this.

## Setup

In [1]:
import warnings; warnings.filterwarnings("ignore")
import sys, time
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("../src").resolve()))
from rogii_wellbore import clean  # noqa: E402

cfg = clean.load_config("../data/interim/clean_config.json")
CLEAN_DIR = Path("../data/interim/clean")
REAL_EVAL_FRAC = 0.73

def tail_mask(n, frac):
    k = int(round(n * frac)); m = np.zeros(n, bool)
    if k: m[n - k:] = True
    return m

def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

def load_pair(wid):
    hz = pd.read_csv(CLEAN_DIR / "train" / f"{wid}__horizontal_well.csv",
                     dtype={"well_id": str}).sort_values("MD").reset_index(drop=True)
    tw = pd.read_csv(CLEAN_DIR / "train" / f"{wid}__typewell.csv",
                     dtype={"well_id": str}).reset_index(drop=True)
    return hz, tw

WELLS = sorted(p.name.split("__")[0]
               for p in (CLEAN_DIR / "train").glob("*__horizontal_well.csv"))
print(f"{len(WELLS)} train wells")

773 train wells


## Engines

**Particle filter** — state is the offset `pos = TVT + Z`; rate (offset drift)
initialized from the median local drift of the last `rate_win` known rows; per
step: momentum + noise on rate, advance pos, recover `TVT = pos − Z` with the
*known* eval-row Z; GR reweights particles via a Gaussian likelihood against the
typewell GR at each particle's implied TVT, with a **per-well sigma** estimated
from the known-zone lateral-vs-typewell misfit (clipped). Systematic resampling
with roughening. Uses **raw-scale GR** (sigma clip is in API units, per the kernel).

**Beam search** — discrete path over the typewell grid (state = index), ±2 index
moves per row, cost = GR misfit / `es` + `mc`·|move|; backtrack the best path.

**Blends** (computed free from stored predictions): `hold` mixes the flat floor;
`pf+beam` mixes the two engines; seed-ensemble averages PF over seeds.

In [2]:
def _prep_well(hz, tw, frac):
    # one-time per-well preparation shared by all variants
    tw_s = tw.sort_values("TVT")
    twt = tw_s["TVT"].values.astype(float)
    twg = tw_s["GR"].ffill().bfill().values.astype(float)      # raw-scale
    tvt = hz["TVT"].values.astype(float)
    Z = hz["Z"].values.astype(float); MD = hz["MD"].values.astype(float)
    gr = pd.Series(hz["GR"].values).interpolate(limit_direction="both")\
           .fillna(np.nanmean(twg)).values
    m = tail_mask(len(hz), frac); kn = np.where(~m)[0]; ev = np.where(m)[0]
    return dict(twt=twt, twg=twg, tvt=tvt, Z=Z, MD=MD, gr=gr, kn=kn, ev=ev)

def run_pf(P, N=500, spread=4.5, MOM=0.998, VN=0.002, PN=0.005,
           rate_win=30, gs_max=60.0, seed=42, RESAMP=0.5, RP=0.1, RR=0.001):
    twt, twg, tvt, Z, MD, gr, kn, ev = (P[k] for k in
        ("twt", "twg", "tvt", "Z", "MD", "gr", "kn", "ev"))
    last = kn[-1]
    tw_at_k = np.interp(tvt[kn], twt, twg)
    gs = float(np.clip(np.nanstd(gr[kn] - tw_at_k), 10.0, gs_max))
    tl = kn[-rate_win:]
    dt = np.diff(tvt[tl]); dz = np.diff(Z[tl]); dm = np.diff(MD[tl]); ok = dm > 0
    ir = float(np.median((dt + dz)[ok] / dm[ok])) if ok.sum() >= 3 else 0.0
    rng = np.random.default_rng(seed)
    pos = (tvt[last] + Z[last]) + spread * rng.standard_normal(N)
    rate = ir + 0.01 * rng.standard_normal(N)
    w = np.ones(N) / N
    out = np.empty(len(ev)); prev = MD[last]
    lo, hi = twt[0] - 100, twt[-1] + 100
    for i, idx in enumerate(ev):
        dmS = max(MD[idx] - prev, 1.0)
        rate = MOM * rate + VN * rng.standard_normal(N)
        pos = pos + rate * dmS + PN * rng.standard_normal(N)
        tvt_p = np.clip(pos - Z[idx], lo, hi); pos = tvt_p + Z[idx]
        g = gr[idx]
        if np.isfinite(g):
            d = (g - np.interp(tvt_p, twt, twg)) / gs
            w = w * np.maximum(np.exp(-0.5 * np.minimum(d * d, 600.0)), 1e-300)
            s = w.sum(); w = w / s if s > 0 else np.ones(N) / N
        if 1.0 / np.sum(w * w) < RESAMP * N:
            ci = np.searchsorted(np.cumsum(w), (np.arange(N) + rng.uniform(0, 1)) / N)
            ci = np.clip(ci, 0, N - 1)
            pos = pos[ci] + RP * rng.standard_normal(N)
            rate = rate[ci] + RR * rng.standard_normal(N)
            w = np.ones(N) / N
        out[i] = np.sum(w * (pos - Z[idx]))
        prev = MD[idx]
    return out

def run_beam(P, BS=10, mc=20.0, es=144.0, smooth=2):
    twt, twg, tvt, MD, gr, kn, ev = (P[k] for k in
        ("twt", "twg", "tvt", "MD", "gr", "kn", "ev"))
    g = gr
    if smooth and smooth > 1:
        g = pd.Series(g).rolling(smooth, min_periods=1, center=True).mean().values
    si = int(np.argmin(np.abs(twt - tvt[kn[-1]])))
    beams = {si: 0.0}; hist = []
    for i in ev:
        gv = g[i]; cand = {}
        for idx, cost in beams.items():
            for d in (-2, -1, 0, 1, 2):
                ni = idx + d
                if ni < 0 or ni >= len(twt): continue
                tot = cost + (gv - twg[ni]) ** 2 / es + mc * abs(d)
                if ni not in cand or tot < cand[ni][0]: cand[ni] = (tot, idx)
        top = sorted(cand.items(), key=lambda kv: kv[1][0])[:BS]
        hist.append({ni: pidx for ni, (c, pidx) in top})
        beams = {ni: c for ni, (c, pidx) in top}
    best = min(beams, key=beams.get); path = [best]
    for hmap in reversed(hist[1:]):
        best = hmap.get(best, best); path.append(best)
    return twt[np.array(path[::-1])]

## The variant grid

One-at-a-time axes off the kernel's base config, plus seed ensembles, beam
configs, and post-hoc blends (free). Heavy runs ≈ 24/well at ~0.3–0.6 s each.
**Start with `N_WELLS = 60` to time it**; the full 773 is roughly 1.5–3 h.

In [3]:
BASE = dict(N=500, spread=4.5, MOM=0.998, VN=0.002, PN=0.005, rate_win=30, gs_max=60.0)

PF_VARIANTS = {"pf_base": dict(BASE)}
for sp in (2.0, 3.0, 8.0, 12.0):
    PF_VARIANTS[f"pf_spread{sp:g}"] = dict(BASE, spread=sp)
for mom in (0.99, 0.995, 0.9995):
    PF_VARIANTS[f"pf_mom{mom:g}"] = dict(BASE, MOM=mom)
for vn in (0.001, 0.005):
    PF_VARIANTS[f"pf_vn{vn:g}"] = dict(BASE, VN=vn)
for n in (300, 1000):
    PF_VARIANTS[f"pf_N{n}"] = dict(BASE, N=n)
for rw in (15, 60):
    PF_VARIANTS[f"pf_rw{rw}"] = dict(BASE, rate_win=rw)
for gm in (40.0, 90.0):
    PF_VARIANTS[f"pf_gs{gm:g}"] = dict(BASE, gs_max=gm)

SEEDS = (42, 7, 2024, 99, 1234)          # seed ensemble of the base config
BEAM_VARIANTS = {
    "beam_cons":  dict(BS=10, mc=20.0, es=144.0, smooth=2),
    "beam_loose": dict(BS=10, mc=8.0,  es=64.0,  smooth=2),
    "beam_vcons": dict(BS=8,  mc=35.0, es=220.0, smooth=1),
}
HOLD_WEIGHTS = (0.05, 0.10, 0.15, 0.20)
PF_BEAM_MIX = (0.3, 0.5)
print(f"{len(PF_VARIANTS)} PF variants + {len(SEEDS)} seeds + {len(BEAM_VARIANTS)} beams"
      f" (+ blends derived free)")

16 PF variants + 5 seeds + 3 beams (+ blends derived free)


In [4]:
SAVE_PRED_VARIANTS = ["pf_spread2", "pf_seedens", "pf_base", "pf_N300", "pf_spread8", "beam_cons"]
PRED_DIR = Path("../data/interim/pf_preds"); PRED_DIR.mkdir(exist_ok=True, parents=True)

def sweep_well(hz, tw, wid, frac=REAL_EVAL_FRAC):
    P = _prep_well(hz, tw, frac)
    tvt, kn, ev = P["tvt"], P["kn"], P["ev"]
    true = tvt[ev]; hold = np.full(len(ev), tvt[kn[-1]])
    out = {"eval_span": float(true.max() - true.min()),
           "n_eval": int(len(ev)), "n_known": int(len(kn)),
           "z_span": float(P["Z"].max() - P["Z"].min()),
           "gr_misfit": float(np.nanstd(P["gr"][kn] - np.interp(tvt[kn], P["twt"], P["twg"]))),
           "floor": rmse(hold, true)}
    preds = {}
    for name, kw in PF_VARIANTS.items():
        p = run_pf(P, **kw); preds[name] = p; out[name] = rmse(p, true)
    seed_preds = [run_pf(P, **BASE, seed=s) for s in SEEDS]
    ens = np.mean(seed_preds, axis=0)
    preds["pf_seedens"] = ens; out["pf_seedens"] = rmse(ens, true)
    for name, kw in BEAM_VARIANTS.items():
        p = run_beam(P, **kw); preds[name] = p; out[name] = rmse(p, true)
    for h in HOLD_WEIGHTS:
        out[f"pf_base_hold{h:g}"] = rmse((1 - h) * preds["pf_base"] + h * hold, true)
        out[f"pf_seedens_hold{h:g}"] = rmse((1 - h) * preds["pf_seedens"] + h * hold, true)
    for bmix in PF_BEAM_MIX:
        pb = (1 - bmix) * preds["pf_seedens"] + bmix * preds["beam_cons"]
        out[f"pf_beam{bmix:g}"] = rmse(pb, true)
        out[f"pf_beam{bmix:g}_hold0.1"] = rmse(0.9 * pb + 0.1 * hold, true)
    # persist predictions + truth for blend/selector work in notebook 7
    np.savez_compressed(PRED_DIR / f"{wid}.npz", true=true.astype(np.float32),
                        hold=hold.astype(np.float32),
                        **{k: preds[k].astype(np.float32) for k in SAVE_PRED_VARIANTS if k in preds})
    return out

def run_sweep(wells, n=None, frac=REAL_EVAL_FRAC):
    wells = wells[:n] if n else wells
    rows = []; t0 = time.time()
    for w_i, wid in enumerate(wells):
        try: hz, tw = load_pair(wid)
        except Exception: continue
        if "TVT" not in hz or hz["TVT"].isna().all(): continue
        nrow = len(hz)
        if int(round(nrow * frac)) < 20 or nrow - int(round(nrow * frac)) < 20: continue
        try:
            rec = sweep_well(hz, tw, wid, frac); rec["well"] = wid; rows.append(rec)
        except Exception:
            continue
        if (w_i + 1) % 20 == 0:
            el = time.time() - t0
            print(f"  ...{w_i+1} wells  [{el/60:.1f} min, {el/(w_i+1):.1f} s/well]")
    return pd.DataFrame(rows)

N_WELLS = None         # timing pass first; set to None for the full 773
sweep = run_sweep(WELLS, n=N_WELLS)
print(f"swept {len(sweep)} wells")

  ...20 wells  [1.2 min, 3.5 s/well]
  ...40 wells  [2.5 min, 3.7 s/well]
  ...60 wells  [3.7 min, 3.7 s/well]
  ...80 wells  [5.0 min, 3.8 s/well]
  ...100 wells  [6.3 min, 3.8 s/well]
  ...120 wells  [7.6 min, 3.8 s/well]
  ...140 wells  [8.9 min, 3.8 s/well]
  ...160 wells  [10.1 min, 3.8 s/well]
  ...180 wells  [11.4 min, 3.8 s/well]
  ...200 wells  [12.6 min, 3.8 s/well]
  ...220 wells  [13.8 min, 3.8 s/well]
  ...240 wells  [15.1 min, 3.8 s/well]
  ...260 wells  [16.4 min, 3.8 s/well]
  ...280 wells  [17.6 min, 3.8 s/well]
  ...300 wells  [18.8 min, 3.8 s/well]
  ...320 wells  [20.2 min, 3.8 s/well]
  ...340 wells  [21.6 min, 3.8 s/well]
  ...360 wells  [22.8 min, 3.8 s/well]
  ...380 wells  [24.1 min, 3.8 s/well]
  ...400 wells  [25.4 min, 3.8 s/well]
  ...420 wells  [26.7 min, 3.8 s/well]
  ...440 wells  [27.8 min, 3.8 s/well]
  ...460 wells  [29.1 min, 3.8 s/well]
  ...480 wells  [30.4 min, 3.8 s/well]
  ...500 wells  [31.6 min, 3.8 s/well]
  ...520 wells  [32.9 min, 3.8 s/wel

## Ranking — RMSE on TVT

In [5]:
arm_cols = [c for c in sweep.columns if c not in ("well", "eval_span")]
overall = sweep[arm_cols].mean().sort_values()
floor_v = overall["floor"]
print("=== OVERALL mean RMSE (top 20) ===")
for k, v in overall.head(20).items():
    tag = "" if k == "floor" else (" <<< beats floor" if v < floor_v else "")
    print(f"  {k:24s} {v:8.3f}{tag}")
print(f"\n(reference: public-notebook class ~7.5; LB leaders ~6.0; floor {floor_v:.2f})")

=== OVERALL mean RMSE (top 20) ===
  pf_seedens_hold0.2         10.776 <<< beats floor
  pf_seedens_hold0.15        10.782 <<< beats floor
  pf_seedens_hold0.1         10.816 <<< beats floor
  pf_beam0.3                 10.853 <<< beats floor
  pf_seedens_hold0.05        10.877 <<< beats floor
  pf_beam0.3_hold0.1         10.943 <<< beats floor
  pf_seedens                 10.967 <<< beats floor
  pf_beam0.5                 11.260 <<< beats floor
  pf_beam0.5_hold0.1         11.390 <<< beats floor
  pf_base_hold0.2            11.511 <<< beats floor
  pf_base_hold0.15           11.589 <<< beats floor
  pf_spread2                 11.649 <<< beats floor
  pf_N1000                   11.679 <<< beats floor
  pf_base_hold0.1            11.697 <<< beats floor
  pf_base_hold0.05           11.836 <<< beats floor
  pf_rw15                    11.890 <<< beats floor
  pf_gs90                    12.011 <<< beats floor
  pf_base                    12.011 <<< beats floor
  pf_gs40                    

In [6]:
print("=== by eval-span bucket (top 6 each) ===")
for lo, hi, lab in [(0, 5, "FLAT <5"), (5, 15, "MED 5-15"), (15, 40, "HIGH 15-40"), (40, 1e9, "XHIGH >40")]:
    sub = sweep[(sweep.eval_span >= lo) & (sweep.eval_span < hi)]
    if not len(sub): continue
    mt = sub[arm_cols].mean().sort_values(); fv = mt["floor"]
    print(f"\n[{lab}] n={len(sub)}  floor={fv:.2f}")
    for k, v in mt.head(6).items():
        if k == "floor": continue
        print(f"   {k:24s} {v:7.3f}{' <<<' if v < fv else ''}")

=== by eval-span bucket (top 6 each) ===

[MED 5-15] n=64  floor=4.97
   pf_beam0.5_hold0.1         4.409 <<<
   pf_beam0.5                 4.526 <<<
   beam_cons                  4.685 <<<
   pf_beam0.3_hold0.1         4.797 <<<
   beam_vcons                 4.805 <<<
   beam_loose                 4.949 <<<

[HIGH 15-40] n=571  floor=11.26
   pf_beam0.3                 9.159 <<<
   pf_seedens_hold0.2         9.176 <<<
   pf_beam0.3_hold0.1         9.201 <<<
   pf_seedens_hold0.15        9.221 <<<
   pf_seedens_hold0.1         9.293 <<<
   pf_seedens_hold0.05        9.392 <<<

[XHIGH >40] n=138  floor=26.28
   gr_misfit                 14.272 <<<
   pf_spread2                19.183 <<<
   pf_seedens                19.203 <<<
   pf_seedens_hold0.05       19.331 <<<
   pf_seedens_hold0.1        19.494 <<<
   pf_seedens_hold0.15       19.692 <<<


In [7]:
# Win-rate + oracle selector over THESE variants (the next notebook's selector ceiling)
valid = sweep[arm_cols]
winner = valid.idxmin(axis=1)
print("=== per-well winner (top 10) ===")
print(winner.value_counts().head(10))
oracle_sel = valid.min(axis=1)
best_name = overall.drop("floor").index[0]
print(f"\nfloor mean:           {sweep['floor'].mean():.2f}")
print(f"best single variant:  {overall.drop('floor').iloc[0]:.2f}  ({best_name})")
print(f"ORACLE selector:      {oracle_sel.mean():.2f}")
print(f"per-well gain over best-single: mean "
      f"{(sweep[best_name] - oracle_sel).mean():.2f} ft")
sweep.to_csv("../data/interim/pf_sweep_results.csv", index=False)
print("\nsaved ../data/interim/pf_sweep_results.csv")

=== per-well winner (top 10) ===
pf_vn0.001      68
pf_vn0.005      65
gr_misfit       61
pf_mom0.99      44
pf_N300         43
pf_spread2      40
pf_mom0.995     38
beam_loose      37
pf_mom0.9995    35
floor           32
Name: count, dtype: int64

floor mean:           13.42
best single variant:  10.78  (pf_seedens_hold0.2)
ORACLE selector:      5.58
per-well gain over best-single: mean 5.19 ft

saved ../data/interim/pf_sweep_results.csv


In [9]:
# --- Regenerate per-well prediction files (correct filenames), resumable ---
PRED_DIR = Path("../data/interim/pf_preds"); PRED_DIR.mkdir(parents=True, exist_ok=True)
stray = PRED_DIR / "0.73.npz"
if stray.exists(): stray.unlink(); print("removed stray 0.73.npz")

REGEN = {"pf_spread2": dict(BASE, spread=2.0), "pf_base": dict(BASE),
         "pf_N300": dict(BASE, N=300), "pf_spread8": dict(BASE, spread=8.0)}

t0 = time.time(); done = skipped = 0
for w_i, wid in enumerate(WELLS):
    assert isinstance(wid, str), f"wid must be a well id string, got {wid!r}"  # guard against this bug class
    out_f = PRED_DIR / f"{wid}.npz"
    if out_f.exists():
        skipped += 1; continue
    try:
        hz, tw = load_pair(wid)
    except Exception:
        continue
    if "TVT" not in hz or hz["TVT"].isna().all(): continue
    nrow = len(hz)
    if int(round(nrow * REAL_EVAL_FRAC)) < 20 or nrow - int(round(nrow * REAL_EVAL_FRAC)) < 20:
        continue
    P = _prep_well(hz, tw, REAL_EVAL_FRAC)
    m = tail_mask(nrow, REAL_EVAL_FRAC)
    P["kn"] = np.where(~m)[0]; P["ev"] = np.where(m)[0]
    true = P["tvt"][P["ev"]]; hold = np.full(len(P["ev"]), P["tvt"][P["kn"][-1]])
    preds = {name: run_pf(P, **kw) for name, kw in REGEN.items()}
    preds["pf_seedens"] = np.mean([run_pf(P, **BASE, seed=s) for s in SEEDS], axis=0)
    preds["beam_cons"] = run_beam(P, **BEAM_VARIANTS["beam_cons"])
    np.savez_compressed(out_f, true=true.astype(np.float32), hold=hold.astype(np.float32),
                        **{k: v.astype(np.float32) for k, v in preds.items()})
    done += 1
    if done % 50 == 0:
        print(f"  {done} regenerated ({skipped} skipped) [{(time.time()-t0)/60:.1f} min]")
print(f"DONE: {done} regenerated, {skipped} already existed [{(time.time()-t0)/60:.1f} min]")

removed stray 0.73.npz
DONE: 0 regenerated, 773 already existed [0.0 min]


## Reading this & next notebook

Three numbers decide the focus: the **best single variant** (is one config enough?),
the **oracle selector** (how much per-well selection adds — if the gap is large,
notebook 7 builds the selector on well features like eval length / Z-span, as the
public kernel does), and the **bucket tables** (whether different buckets want
different variants — the selector's natural bins). If `pf_seedens_hold*` or
`pf_beam*` blends lead, the production pipeline is an ensemble, and notebook 7
should also add the LGBM branch the public solution blends at 0.45.

Run the timing pass, then set `N_WELLS = None` for the full 773 before concluding.